<a href="https://colab.research.google.com/github/minyi-k03/LargeLanguageModel/blob/Project-Based-Learning(PBL)/Llama3_2_vision_vqa_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Llama 3.2-Vision 모델 한국어 VQA 성능 테스트하기
## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )
## Reference : https://huggingface.co/meta-llama/Llama-3.2-11B-Vision-Instruct
## [시각정보 기반 질의응답] 데이터 다운로드 : https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=data&dataSetSn=104

In [ ]:
!nvidia-smi

# 라이브러리 설치

In [ ]:
# [Cell 1] 라이브러리 설치
import torch

# 1. GPU 확인
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("GPU가 없습니다. 런타임 유형을 T4로 변경하세요.")

print("\nInstalling Libraries for Llama-3.2 Vision...")

# 2. 필수 라이브러리 설치
!pip install -U "transformers>=4.45.0" "accelerate" "bitsandbytes" "huggingface_hub"


# Llama 3.2 모델 불러오기

In [ ]:
# [Cell 2] Llama-3.2-11B-Vision 로드 (4-bit 양자화)
import os
import torch
from transformers import MllamaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# 1. 토큰 설정
os.environ['HF_TOKEN'] = "Input Your Token"

model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"

# 2. 4-bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Loading Vision Model: {model_id}...")

# 3. 모델 로드 (Vision 모델 전용 클래스 사용)
model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config, # 4-bit 적용
    device_map="auto",
    torch_dtype=torch.float16
)

# 4. 프로세서 로드 (이미지+텍스트 처리용)
processor = AutoProcessor.from_pretrained(model_id)

print(f"Model Loaded on {model.device}")

In [ ]:
import requests
from PIL import Image

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/0052a70beed5bf71b92610a43a52df6d286cd5f3/diffusers/rabbit.jpg"
image = Image.open(requests.get(url, stream=True).raw)

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "If I had to write a haiku for this one, it would be: "}
    ]}
]
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt"
).to(model.device)

output = model.generate(**inputs, max_new_tokens=30)
print(processor.decode(output[0]))

In [ ]:
image

In [ ]:
messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "이 이미지를 묘사해줘"}
    ]}
]
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt"
).to(model.device)

output = model.generate(**inputs, max_new_tokens=256)
print(processor.decode(output[0]))

In [ ]:
messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "이미지 안에 토끼의 코트의 색깔은 무슨 색이야?"}
    ]}
]
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt"
).to(model.device)

output = model.generate(**inputs, max_new_tokens=256)
print(processor.decode(output[0]))

# [시각정보 기반 질의응답] 데이터 불러오기

In [ ]:
# 테스트용 이미지 3개 불러오기

# NIA_dataset02_000000777750.jpg
# NIA_dataset02_000000777856.jpg
# NIA_dataset02_000000777936.jpg

## 테스트 이미지 3장 읽어오기

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# 이미지 파일 경로
test_image_1_filename = "NIA_dataset02_000000777750.jpg"

# 이미지 읽기
test_image_1 = Image.open(test_image_1_filename)
test_image_1

In [ ]:
# 이미지 파일 경로
test_image_2_filename = "NIA_dataset02_000000777856.jpg"

# 이미지 읽기
test_image_2 = Image.open(test_image_2_filename)
test_image_2

In [ ]:
# 이미지 파일 경로
test_image_3_filename = "NIA_dataset02_000000777936.jpg"

# 이미지 읽기
test_image_3 = Image.open(test_image_3_filename)
test_image_3

## 레이블 json 파일들 읽어오기

In [ ]:
import json

# JSON 파일 경로
images_json_filename = 'images.json'

# JSON 파일 읽기
with open(images_json_filename, 'r', encoding='utf-8') as file:
    images_json_data = json.load(file)
images_json_data

In [ ]:
def get_image_id_by_filename(data, image_filename):
    for image_info in data.get('images', []):
        if image_info.get('image') == image_filename:
            return image_info.get('image_id')
    return None

In [ ]:
test_image_1_image_id = get_image_id_by_filename(images_json_data, test_image_1_filename)
test_image_1_image_id

In [ ]:
test_image_2_image_id = get_image_id_by_filename(images_json_data, test_image_2_filename)
test_image_2_image_id

In [ ]:
test_image_3_image_id = get_image_id_by_filename(images_json_data, test_image_3_filename)
test_image_3_image_id

In [ ]:
# JSON 파일 경로
question_json_filename = 'question.json'

# JSON 파일 읽기
with open(question_json_filename, 'r', encoding='utf-8') as file:
    question_json_data = json.load(file)
question_json_data

In [ ]:
def get_questions_by_image_id(data, image_id):
    questions = [{'question_id': q['question_id'], 'question': q['question']} for q in data.get('questions', []) if q.get('image_id') == image_id]
    return questions

In [ ]:
test_image_1_questions = get_questions_by_image_id(question_json_data, test_image_1_image_id)
test_image_1_questions

In [ ]:
test_image_2_questions = get_questions_by_image_id(question_json_data, test_image_2_image_id)
test_image_2_questions

In [ ]:
test_image_3_questions = get_questions_by_image_id(question_json_data, test_image_3_image_id)
test_image_3_questions

In [ ]:
#각 이미지 질문에 대한 정
# JSON 파일 경로
annotation_json_filename = 'annotation.json'

# JSON 파일 읽기
with open(annotation_json_filename, 'r', encoding='utf-8') as file:
    annotation_json_data = json.load(file)
annotation_json_data

In [ ]:
def get_answer_by_question_id(data, question_id):
    for annotation in data.get('annotations', []):
        if annotation.get('question_id') == question_id:
            return {'question_id': annotation.get('question_id'), 'multiple_choice_answer': annotation.get('multiple_choice_answer')}
    return None

In [ ]:
test_image_1

In [ ]:
test_image_1_questions

In [ ]:
test_image_1_answer = get_answer_by_question_id(annotation_json_data, '777750001')
test_image_1_answer

In [ ]:
test_image_1_answer = get_answer_by_question_id(annotation_json_data, '777750002')
test_image_1_answer

In [ ]:
test_image_1_answer = get_answer_by_question_id(annotation_json_data, '777750003')
test_image_1_answer

In [ ]:
test_image_1_answer = get_answer_by_question_id(annotation_json_data, '777750004')
test_image_1_answer

In [ ]:
test_image_1_answer = get_answer_by_question_id(annotation_json_data, '777750005')
test_image_1_answer

In [ ]:
test_image_2

In [ ]:
test_image_2_questions = get_questions_by_image_id(question_json_data, test_image_2_image_id)
test_image_2_questions

In [ ]:
# 식탁 위에 몇 개의 병이 있습니까?
test_image_2_answer = get_answer_by_question_id(annotation_json_data, '777856004')
test_image_2_answer

In [ ]:
# 식탁 위에 놓인 음식은 슬로우 푸드입니까, 패스트 푸드입니까?
test_image_2_answer = get_answer_by_question_id(annotation_json_data, '799868002')
test_image_2_answer

In [ ]:
# 식탁에 놓인 음식은 열량이 높은 편인가요?
test_image_2_answer = get_answer_by_question_id(annotation_json_data, '799868005')
test_image_2_answer

In [ ]:
test_image_3

In [ ]:
test_image_3_questions = get_questions_by_image_id(question_json_data, test_image_3_image_id)
test_image_3_questions

In [ ]:
# 주방에서 요리를 하는 사람은 몇 명입니까?
test_image_3_answer = get_answer_by_question_id(annotation_json_data, '777936003')
test_image_3_answer

In [ ]:
# 요리하고 있는 사람의 앞치마는 무슨 색입니까?
test_image_3_answer = get_answer_by_question_id(annotation_json_data, '777936005')
test_image_3_answer

In [ ]:
def get_llama_response(image_filename, question):
    # Path to your image
    image_path = image_filename
    image = Image.open(image_path)

    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": f'{question}'}
        ]}
    ]

    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(model.device)

    # do_sample=False로 설정
    output = model.generate(**inputs, max_new_tokens=256, do_sample=False) #SoftMax Regression 값이 큰것만 고르게끔 지정
    #print(processor.decode(output[0]))
    full_text = processor.decode(output[0])

    # 분리 기준 문자열
    delimiter = "<|start_header_id|>assistant<|end_header_id|>\n\n"

    # 분리하여 해당 부분 뒤의 문자열만 추출
    extracted_text = full_text.split(delimiter, 1)[1]  # split()의 두 번째 인수로 1을 지정하여 첫 번째 분리만 수행
    # ".<|eot_id|>" 제거
    extracted_text = extracted_text.replace(".<|eot_id|>", "")

    return extracted_text

In [ ]:
test_image_1

In [ ]:
test_image_1_questions

In [ ]:
get_llama_response(test_image_1_filename, '접시에 담겨 있는 것은 무엇입니까?')
# 정답 : '치즈'

In [ ]:
get_llama_response(test_image_1_filename, '테이블 왼쪽에 접시가 쌓여 있습니까?')
# 정답 : '예'

In [ ]:
get_llama_response(test_image_1_filename, '접시 위에 녹색 야채는 무엇입니까?')
# 정답 : '애플민트'

In [ ]:
get_llama_response(test_image_1_filename, '테이블 위에 유리잔이 있습니까?')
# 정답 : '예'

In [ ]:
get_llama_response(test_image_1_filename, '접시 위에 원형 치즈가 있습니까?')
# 정답 : '예'

In [ ]:
test_image_2

In [ ]:
test_image_2_questions

In [ ]:
get_llama_response(test_image_2_filename, '식탁 위에 몇 개의 병이 있습니까?')
# 정답 : '2'

In [ ]:
#정답과 틀림
get_llama_response(test_image_2_filename, '식탁 위에 놓인 음식은 슬로우 푸드입니까, 패스트 푸드입니까?')
# 정답 : '패스트 푸드'

In [ ]:
get_llama_response(test_image_2_filename, '식탁에 놓인 음식은 열량이 높은 편인가요?')
# 정답 : '에'

In [ ]:
test_image_3

In [ ]:
test_image_3_questions

In [ ]:
get_llama_response(test_image_3_filename, '주방에서 요리를 하는 사람은 몇 명입니까?')
# 정답 : '2'

In [ ]:
get_llama_response(test_image_3_filename, '요리하고 있는 사람의 앞치마는 무슨 색입니까?')
# 정답 : '흰색'